# Blindference Agent — Getting Started

This notebook walks you through submitting confidential inference requests using the Blindference Agent SDK.

## Two modes:
1. **Real mode** — End-to-end encryption with CoFHE (requires private key + CoFHE bridge)
2. **Mock mode** — No encryption, plaintext prompts (for demos and testing)

The notebook starts with **real mode** instructions, then falls back to **mock mode** for quick demos.

In [ ]:
# Cell 1: Setup Check
import sys
import shutil
import os
from pathlib import Path

print('🔍 Blindference Agent Setup Check')
print('=' * 50)

# 1. Python version
py_version = sys.version_info
ok = py_version >= (3, 10)
print(f'Python {py_version.major}.{py_version.minor}.{py_version.micro} {"✅" if ok else "❌ (need ≥3.10)"}')

# 2. Node.js installed
node = shutil.which('node')
print(f'Node.js {"✅" if node else "❌ (install Node.js for CoFHE bridge)"} — {node or "not found"}')

# 3. npm dependencies installed
nm = Path('node_modules')
cofhe_sdk = nm / '@cofhe' / 'sdk'
viem = nm / 'viem'
npm_ok = cofhe_sdk.exists() and viem.exists()
print(f'npm deps {"✅" if npm_ok else "❌ (run: npm install)"}')

# 4. .env file exists
env_path = Path('.env')
env_ok = env_path.exists()
print(f'.env file {"✅" if env_ok else "❌ (create .env from .env.example)"}')

# 5. Agent SDK installed
try:
    from blindference_agent import BlindferenceAgent
    print('Agent SDK ✅')
except ImportError:
    print('Agent SDK ❌ (run: pip install -e .)')

print()
if not all([ok, node, npm_ok, env_ok]):
    print('⚠️  Some checks failed. Fix them before real mode, or use mock mode below.')
else:
    print('🚀 All checks passed! Ready for real mode.')

## Cell 2: Simple Inference

### Real Mode (end-to-end encryption)

Uncomment and run this cell after setting up `.env` with your keys:

In [ ]:
# REAL MODE — uncomment after setting up .env
# import asyncio
# import os
# from blindference_agent import BlindferenceAgent
# 
# async def real_inference():
#     agent = BlindferenceAgent(
#         icl_url=os.environ.get('BLF_ICL_URL', 'https://icl.blindference.xyz'),
#         cofhe_rpc=os.environ.get('BLF_COFHE_RPC', ''),
#         private_key=os.environ.get('BLF_PRIVATE_KEY', ''),
#     )
#     result = await agent.inference(
#         prompt='Explain quantum computing in one sentence',
#         model_id='groq:llama-3.3-70b-versatile',
#         verifier_count=2,
#     )
#     print('Result:', result.text)
#     print(f'Leader: {result.leader_address}')
#     print(f'Verifiers: {result.verifier_addresses}')
#     await agent.close()
# 
# asyncio.run(real_inference())

### Mock Mode (no encryption — for demos)

Run this cell immediately to see the API in action without any setup:

In [ ]:
# MOCK MODE — works without keys or CoFHE bridge
import asyncio
from blindference_agent import BlindferenceAgent

async def mock_inference():
    agent = BlindferenceAgent(
        icl_url='https://icl.blindference.xyz',
        mock=True,  # <-- enables mock mode
    )
    result = await agent.inference(
        prompt='Explain quantum computing in one sentence',
        model_id='groq:llama-3.3-70b-versatile',
        verifier_count=2,
    )
    print('=' * 60)
    print('Result:', result.text)
    print('=' * 60)
    print(f'Model:     {result.model_id}')
    print(f'Request:   {result.request_id}')
    print(f'Task:      {result.task_id}')
    print(f'Leader:    {result.leader_address}')
    print(f'Verifiers: {result.verifier_addresses}')
    print(f'Commit:    {result.commitment_hash}')
    await agent.close()

await mock_inference()

## Cell 3: Streaming Progress

Watch the quorum execute in real time with a progress bar:

In [ ]:
# Streaming with tqdm progress bar
import asyncio
from tqdm.notebook import tqdm
from blindference_agent import BlindferenceAgent

async def streaming_demo():
    agent = BlindferenceAgent(icl_url='https://icl.blindference.xyz', mock=True)
    
    # Submit and immediately stream progress
    request = await agent.submit(
        prompt='What is the capital of France?',
        model_id='gemini:gemini-2.5-flash',
        verifier_count=2,
    )
    
    print(f'Request ID: {request.request_id}')
    print('Streaming quorum progress...\n')
    
    pbar = tqdm(total=5, desc='Execution Trace')
    step_names = {'quorum': 'Building Quorum', 'leader': 'Leader Inference', 
                  'verifier': 'Verifier Replay', 'onchain': 'On-Chain Commit',
                  'decrypt': 'Local Decrypt'}
    
    async for status in agent.stream_status(request.request_id, interval=1.0):
        step_label = step_names.get(status.step, status.step)
        pbar.set_description(step_label)
        
        if status.status == 'ASSIGNED':
            pbar.update(1)
        elif status.status == 'EXECUTING':
            pbar.update(1)
        elif status.status == 'VERIFYING':
            pbar.update(1)
        elif status.status == 'ACCEPTED':
            pbar.update(2)
            pbar.close()
            print('\n✅ Inference complete!')
            break
        elif status.status in ('REJECTED', 'DISPUTED'):
            pbar.close()
            print(f'\n❌ Failed: {status.status}')
            break
    
    await agent.close()

await streaming_demo()

## Cell 4: Batch Inference

Submit 3 prompts concurrently and collect all results:

In [ ]:
# Batch inference — 3 prompts at once
import asyncio
from blindference_agent import BlindferenceAgent

async def batch_inference():
    agent = BlindferenceAgent(icl_url='https://icl.blindference.xyz', mock=True)
    
    prompts = [
        'What is machine learning?',
        'Explain neural networks briefly',
        'What is overfitting in ML?',
    ]
    
    print(f'Submitting {len(prompts)} prompts concurrently...\n')
    
    # Submit all requests concurrently
    requests = await asyncio.gather(*[
        agent.submit(prompt, 'groq:llama-3.3-70b-versatile', verifier_count=2)
        for prompt in prompts
    ])
    
    # Wait for all to complete
    results = []
    for req in requests:
        while True:
            status = await agent.icl.get_status(req.request_id)
            if status.status == 'ACCEPTED':
                result = await agent._decrypt_result(status, req.model_id)
                results.append(result)
                break
            await asyncio.sleep(1.0)
    
    print('=' * 60)
    for i, result in enumerate(results, 1):
        print(f'\nPrompt {i}: {prompts[i-1]}')
        print(f'Result: {result.text[:100]}...')
        print(f'Request: {result.request_id}')
    print('=' * 60)
    
    await agent.close()

await batch_inference()

## Cell 5: Inspect Result Metadata

Explore the full `InferenceResult` object:

In [ ]:
# Inspect InferenceResult metadata
import asyncio
from blindference_agent import BlindferenceAgent

async def inspect_result():
    agent = BlindferenceAgent(icl_url='https://icl.blindference.xyz', mock=True)
    
    result = await agent.inference(
        prompt='What is the Pythagorean theorem?',
        model_id='groq:llama-3.3-70b-versatile',
        verifier_count=2,
    )
    
    print('🔍 InferenceResult Fields:')
    print('-' * 50)
    for field, value in result.__dict__.items():
        display = value
        if isinstance(value, str) and len(value) > 50:
            display = value[:50] + '...'
        print(f'{field:25s}: {display}')
    
    print()
    print('📊 Summary:')
    print(f'  Text length: {len(result.text)} chars')
    print(f'  Verifiers: {len(result.verifier_addresses)}')
    print(f'  Commitment: {result.commitment_hash[:20]}...' if result.commitment_hash else '  No commitment')
    
    await agent.close()

await inspect_result()

## Cell 6: Quorum Information

Preview the quorum before submitting:

In [ ]:
# Quorum preview — see who will execute your request
import asyncio
from blindference_agent import BlindferenceAgent

async def quorum_info():
    agent = BlindferenceAgent(icl_url='https://icl.blindference.xyz', mock=True)
    
    quorum = await agent.icl.quorum_preview(
        model_id='groq:llama-3.3-70b-versatile',
        verifier_count=2,
        min_tier=0,
    )
    
    print('👑 Leader Node:')
    print(f'  Address: {quorum.leader.address}')
    print(f'  Tier:    {quorum.leader.tier}')
    print(f'  Rep:     {quorum.leader.reputation_score}')
    
    print()
    print('🔍 Verifier Nodes:')
    for i, v in enumerate(quorum.verifiers, 1):
        print(f'  {i}. Address: {v.address}')
        print(f'      Tier:    {v.tier}')
        print(f'      Rep:     {v.reputation_score}')
    
    await agent.close()

await quorum_info()

## Cell 7: Interactive Chat Loop

A simple REPL chat with the Blindference network:

In [ ]:
# Interactive chat loop
import asyncio
from blindference_agent import BlindferenceAgent

async def chat_loop():
    agent = BlindferenceAgent(icl_url='https://icl.blindference.xyz', mock=True)
    
    print('🤖 Blindference Agent Chat')
    print('Type your question or "exit" to quit')
    print('=' * 60)
    
    while True:
        user_input = input('You: ')
        if user_input.lower() in ('exit', 'quit', 'q'):
            break
        
        print('  Thinking...')
        result = await agent.inference(
            prompt=user_input,
            model_id='groq:llama-3.3-70b-versatile',
            verifier_count=2,
        )
        
        print(f'Agent: {result.text}')
        print(f'  [Leader: {result.leader_address[:20]}..., Verifiers: {len(result.verifier_addresses)}]')
        print()
    
    print('Goodbye! 👋')
    await agent.close()

await chat_loop()